In [ ]:
from skimage.io import imread, imsave
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Parameters
intensity_threshold_percentile = 90
nuclearization_threshold = 1.65  # Example constant value, adjust as needed

# Define the directory path containing the images
image_directory = r'G:\Chia_Ling_Yeast_Live_imaging\20241004_e_ek_ep_ekp'
label_directory = r'G:\Chia_Ling_Yeast_Live_imaging\20241004_e_ek_ep_ekp'
output_directory = r'G:\Chia_Ling_Yeast_Live_imaging\20241004_e_ek_ep_ekp\V8_output_p90_1p65'


# Ensure the directory exists
os.makedirs(output_directory, exist_ok=True)

# Define the specific naming patterns for images and labels
image_pattern = "_intensity.tif"
filter_label_pattern = "_filtered_label.tif"

# List all image files in the directory that match the specified naming pattern
image_files = [f for f in os.listdir(image_directory) if f.endswith(image_pattern) and os.path.isfile(os.path.join(image_directory, f))]
label_files = [f for f in os.listdir(label_directory) if f.endswith(filter_label_pattern) and os.path.isfile(os.path.join(label_directory, f))]

# Sort files to ensure matching order of images and labels if necessary
image_files.sort()
label_files.sort()

# Iterate over the files and load them
for img_file, label_file in zip(image_files, label_files):
    img_path = os.path.join(image_directory, img_file)
    label_path = os.path.join(label_directory, label_file)
    
    img_time_series = imread(img_path)
    img_label_time_series = imread(label_path)

    base_filename, file_extension = os.path.splitext(img_file)
    base_filename = base_filename.replace('_intensity', '')

    # Initialize a list to store nuclearization ratio images for all time points
    nuclearization_ratio_images = []
    nuclearization_ratio_time_series = {}

    for t in range(img_label_time_series.shape[0]):
        label_image = img_label_time_series[t, :, :]
        intensity_image = img_time_series[t, :, :]
        nuclearization_ratio = {}

        nuclearization_ratio_image = np.zeros_like(label_image, dtype=np.float32)

        for label in np.unique(label_image):
            if label == 0:
                continue

            mask = label_image == label
            pixel_intensities = intensity_image[mask]
            median_intensity = np.median(pixel_intensities)
            percentile_95 = np.percentile(pixel_intensities, 95)
            top_5_percent = pixel_intensities[pixel_intensities >= percentile_95]

            if len(top_5_percent) > 0:
                nuclearization_ratio[label] = np.sum(top_5_percent) / len(top_5_percent) / median_intensity
                nuclearization_ratio_image[mask] = nuclearization_ratio[label]

        nuclearization_ratio_images.append(nuclearization_ratio_image)
        nuclearization_ratio_time_series[t] = nuclearization_ratio

    nuclearization_ratio_stack = np.stack(nuclearization_ratio_images, axis=0)
    parametric_image_filename = os.path.join(output_directory, f"{base_filename}_nuclearization_ratio_stack.tif")
    imsave(parametric_image_filename, nuclearization_ratio_stack)

    df_nuclearization_ratio = pd.DataFrame.from_dict(nuclearization_ratio_time_series)
    df_nuclearization_ratio_transposed = df_nuclearization_ratio.transpose()

    df_nuclearization_ratio['row_mean'] = df_nuclearization_ratio.mean(axis=1)
    lower_bound = df_nuclearization_ratio['row_mean'].quantile(0.05)
    upper_bound = df_nuclearization_ratio['row_mean'].quantile(0.95)
    df_filtered = df_nuclearization_ratio[(df_nuclearization_ratio['row_mean'] > lower_bound) & (df_nuclearization_ratio['row_mean'] < upper_bound)]
    df_filtered = df_filtered.drop(columns=['row_mean'])

    outlier_labels = set(df_nuclearization_ratio.index) - set(df_filtered.index)

    # Calculate the average intensity for each label across all time frames
    average_intensity_per_label = {}

    for t in range(img_label_time_series.shape[0]):
        label_image = img_label_time_series[t, :, :]
        intensity_image = img_time_series[t, :, :]

        for label in np.unique(label_image):
            if label == 0:
                continue

            mask = label_image == label
            pixel_intensities = intensity_image[mask]

            if label not in average_intensity_per_label:
                average_intensity_per_label[label] = []

            average_intensity_per_label[label].append(np.mean(pixel_intensities))

    # Calculate the overall average intensity for each label
    average_intensity_per_label = {label: np.mean(intensities) for label, intensities in average_intensity_per_label.items()}

    # Determine the threshold for the top 10% average intensities
    intensity_values = list(average_intensity_per_label.values())
    intensity_threshold = np.percentile(intensity_values, intensity_threshold_percentile)

    # Identify labels to remove based on the top 5% average intensity
    labels_to_remove = {label for label, avg_intensity in average_intensity_per_label.items() if avg_intensity > intensity_threshold}

    # Combine with existing outlier labels
    outlier_labels.update(labels_to_remove)

    # Proceed with filtering the label images as before
    final_label_images = []
    for t in range(img_label_time_series.shape[0]):
        label_image = img_label_time_series[t, :, :]
        final_label_image = np.copy(label_image)

        for label in outlier_labels:
            final_label_image[label_image == label] = 0

        final_label_images.append(final_label_image)

    final_nuclearization_ratio_images = []
    for t in range(img_label_time_series.shape[0]):
        label_image = final_label_images[t]
        intensity_image = img_time_series[t, :, :]
        nuclearization_ratio_image = np.zeros_like(label_image, dtype=np.float32)

        for label in np.unique(label_image):
            if label == 0:
                continue

            mask = label_image == label
            pixel_intensities = intensity_image[mask]
            median_intensity = np.median(pixel_intensities)
            percentile_95 = np.percentile(pixel_intensities, 95)
            top_5_percent = pixel_intensities[pixel_intensities >= percentile_95]

            if len(top_5_percent) > 0:
                nuclearization_ratio = np.sum(top_5_percent) / len(top_5_percent) / median_intensity
                nuclearization_ratio_image[mask] = nuclearization_ratio

        final_nuclearization_ratio_images.append(nuclearization_ratio_image)

    final_label_stack = np.stack(final_label_images, axis=0)
    final_parametric_image_stack = np.stack(final_nuclearization_ratio_images, axis=0)

    final_label_filename = os.path.join(output_directory, f"{base_filename}_final_labels.tif")
    final_parametric_image_filename = os.path.join(output_directory, f"{base_filename}_final_nuclearization_ratio_stack.tif")

    imsave(final_label_filename, final_label_stack)
    imsave(final_parametric_image_filename, final_parametric_image_stack)

    # Exclude T1 and T2 from the DataFrame
    df_filtered_excluding_T1_T2 = df_filtered.iloc[:, 2:]  # Exclude the first two columns

    # Use nuclearization_threshold for masking
    masked_df_filtered_excluding_T1_T2 = df_filtered_excluding_T1_T2.mask(
        df_filtered_excluding_T1_T2 <= nuclearization_threshold, 0
    )
    final_masked_df_filtered_excluding_T1_T2 = masked_df_filtered_excluding_T1_T2.mask(
        masked_df_filtered_excluding_T1_T2 > nuclearization_threshold, 1
    )

    # Calculate the nuclearization_score excluding T1 and T2
    result_df_excluding_T1_T2 = df_filtered_excluding_T1_T2.multiply(final_masked_df_filtered_excluding_T1_T2)
    row_sums_excluding_T1_T2 = result_df_excluding_T1_T2.sum(axis=1)

    positive_duration_excluding_T1_T2 = final_masked_df_filtered_excluding_T1_T2.apply(
        lambda row: (row == 1.0).sum(), axis=1
    )

    total_time_frame_excluding_T1_T2 = len(final_masked_df_filtered_excluding_T1_T2.columns)

    nuclearization_score_excluding_T1_T2 = row_sums_excluding_T1_T2.divide(
        positive_duration_excluding_T1_T2
    ).multiply(positive_duration_excluding_T1_T2).divide(total_time_frame_excluding_T1_T2)

    print(nuclearization_score_excluding_T1_T2)

    # Calculate the average nuclearization ratio for each cell ID
    df_filtered['Average'] = df_filtered.mean(axis=1)

    # Export filtered nuclearization ratios to Excel with labels
    excel_filename = os.path.join(output_directory, f"{base_filename}_filtered_nuclearization_ratios.xlsx")
    df_filtered.to_excel(excel_filename, index_label='Cell ID', header=[f'T- {i+1}' for i in range(df_filtered.shape[1] - 1)] + ['Average'])

    # Save the nuclearization score to an Excel file
    excel_filename_excluding_T1_T2 = os.path.join(output_directory, f'{base_filename}_score_excluding_T1_T2.xlsx')
    nuclearization_score_df_excluding_T1_T2 = nuclearization_score_excluding_T1_T2.to_frame(name='Score')
    nuclearization_score_df_excluding_T1_T2.to_excel(excel_filename_excluding_T1_T2, index_label='Cell-ID')

    # Plotting
    num_rows = df_filtered_excluding_T1_T2.shape[0]
    plt.figure(figsize=(20, 10))  # Adjust height based on number of rows
    plt.plot(df_filtered_excluding_T1_T2.transpose())
    plt.savefig(os.path.join(output_directory, f'{base_filename}_Plot_Final.png'), dpi=300)

    # Heatmap
    plt.figure(figsize=(6, num_rows * 0.02))  # Adjust height based on number of rows
    sns.heatmap(df_filtered_excluding_T1_T2, cmap="YlGnBu", annot=False, cbar=True, vmin=2, vmax=4)

    plt.title('Heat Map of Nuclearization Ratio Over Time (Filtered)')
    plt.xlabel('Time Periods')
    plt.ylabel('Cell-ID')
    plt.xticks(rotation=45)
    plt.tight_layout()

    plt.savefig(os.path.join(output_directory, f'{base_filename}_HeatMap_Final.png'), dpi=300)
    plt.show()

    plt.figure(figsize=(6, num_rows * 0.02)) # Adjust height based on number of rows
    # Generate a heatmap
    sns.heatmap(final_masked_df_filtered_excluding_T1_T2, cmap="YlGnBu", annot=False, cbar=True, vmin=0, vmax=1)
    plt.title('Nuclearization Signal')
    plt.xlabel('Time Periods')
    plt.ylabel('Cell-ID')
    plt.xticks(rotation=45)  # Rotate x-axis labels for better readability
    plt.tight_layout()
    plt.savefig(os.path.join(output_directory, f'{base_filename}_Nuclear_Signal_Final.png'), dpi=300)